In [ ]:
import os
from dvpio.read.image import read_czi
import spatialdata as sd
from spatialdata.models import Image2DModel

display_downsample = 8

czi_dir = r"/Users/rct905/Documents/MaxPlanck_HE_lung_skin/selected/"
czi_skin1a = r"270426_10168_2219_skin1a.czi"
czi_lung2a = r"270426_C15_31186_lung2a.czi"
czi_lung5a = r"270426_E1163_19_lung5a.czi"

czi_path = os.path.join(czi_dir, czi_lung5a)

img = read_czi(czi_path, channels = 0)
image_model = Image2DModel.parse(img)
sdata = sd.SpatialData(
    images={"image": image_model}
)
type(sdata)

spatialdata._core.spatialdata.SpatialData

In [31]:
img = sdata.images[list(sdata.images.keys())[0]]
img

<xarray.DataArray 'image' (c: 3, y: 93440, x: 161310)> Size: 90GB
dask.array<concatenate, shape=(3, 93440, 161310), dtype=uint16, chunksize=(3, 10000, 10000), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) int64 24B 0 1 2
  * y        (y) float64 748kB 0.5 1.5 2.5 3.5 ... 9.344e+04 9.344e+04 9.344e+04
  * x        (x) float64 1MB 0.5 1.5 2.5 3.5 ... 1.613e+05 1.613e+05 1.613e+05
Attributes:
    transform:  {'global': Identity }

In [32]:
fullres = sdata.images["image"]

lowres = fullres.coarsen(
    {fullres.dims[-2]: display_downsample, fullres.dims[-1]: display_downsample},
    boundary="trim",
).mean()

sdata.images["image_vis"] = Image2DModel.parse(lowres)

import random
from shapely.geometry import box
import geopandas as gpd
from spatialdata.models import ShapesModel
import numpy as np

H = img.shape[1]  # image height
W = img.shape[2]  # image width
H = H // display_downsample
W = W // display_downsample

tile_size = 224 // display_downsample  # tile size in low-res coordinates
n_tiles = 20

tiles = []

# your sampling region
x_min, x_max = int(H * 0.3), int(H * 0.7)
y_min, y_max = int(W * 0.3), int(W * 0.7)

max_attempts = 10000
attempts = 0

while len(tiles) < n_tiles and attempts < max_attempts:
    attempts += 1

    x = random.randint(x_min, x_max - tile_size)
    y = random.randint(y_min, y_max - tile_size)

    new_tile = box(x, y, x + tile_size, y + tile_size)

    # check overlap
    if any(new_tile.intersects(existing) for existing in tiles):
        continue

    tiles.append(new_tile)

print(f"Generated {len(tiles)} non-overlapping tiles in {attempts} attempts")

tiles_gdf = gpd.GeoDataFrame(
    {"tile_id": [f"tile_{i}" for i in range(len(tiles))]},
    geometry=tiles,
)

sdata.shapes["tiles"] = ShapesModel.parse(tiles_gdf)

polygons = [
    np.array(g.exterior.coords)
    for g in tiles_gdf.geometry
    if g.geom_type == "Polygon"
]

Generated 20 non-overlapping tiles in 20 attempts


In [33]:
tiles_gdf

,tile_id,geometry
0,tile_0,"POLYGON ((5445 7863, 5445 7891, 5417 7891, 541..."
1,tile_1,"POLYGON ((6398 7537, 6398 7565, 6370 7565, 637..."
2,tile_2,"POLYGON ((6495 14063, 6495 14091, 6467 14091, ..."
3,tile_3,"POLYGON ((3555 8546, 3555 8574, 3527 8574, 352..."
4,tile_4,"POLYGON ((4370 12177, 4370 12205, 4342 12205, ..."
5,tile_5,"POLYGON ((7654 8964, 7654 8992, 7626 8992, 762..."
6,tile_6,"POLYGON ((5701 9842, 5701 9870, 5673 9870, 567..."
7,tile_7,"POLYGON ((7658 9606, 7658 9634, 7630 9634, 763..."
8,tile_8,"POLYGON ((7866 10754, 7866 10782, 7838 10782, ..."
9,tile_9,"POLYGON ((7083 10050, 7083 10078, 7055 10078, ..."


In [34]:
from spatialdata.models import ShapesModel

sdata.shapes["tiles"] = ShapesModel.parse(tiles_gdf)

In [35]:
sdata

SpatialData object
├── Images
│     ├── 'image': DataArray[cyx] (3, 93440, 161310)
│     └── 'image_vis': DataArray[cyx] (3, 11680, 20163)
└── Shapes
      └── 'tiles': GeoDataFrame shape: (20, 2) (2D shapes)
with coordinate systems:
    ▸ 'global', with elements:
        image (Images), image_vis (Images), tiles (Shapes)

In [ ]:
import napari

viewer = napari.Viewer()

viewer.add_image(
    sdata.images["image_vis"].data,
    channel_axis=0,
    name="WSI_preview",
)
viewer.add_shapes(
    polygons,
    shape_type="polygon",
    edge_color="red",
    face_color="red",
)
napari.run()

Traceback (most recent call last):
  File "/opt/anaconda3/envs/MaxPlanck/lib/python3.11/site-packages/napari/_qt/qt_main_window.py", line 638, in closeEvent
    quit_app_()
  File "/opt/anaconda3/envs/MaxPlanck/lib/python3.11/site-packages/napari/_qt/qt_event_loop.py", line 291, in quit_app
    v.close()
  File "/opt/anaconda3/envs/MaxPlanck/lib/python3.11/site-packages/napari/viewer.py", line 276, in close
    self.window.close()
  File "/opt/anaconda3/envs/MaxPlanck/lib/python3.11/site-packages/napari/_qt/qt_main_window.py", line 1866, in close
    self._qt_viewer.close()
RuntimeError: wrapped C/C++ object of type QtViewer has been deleted
_close_app() missing 1 required positional argument: 'window'
Traceback (most recent call last):
  File "/opt/anaconda3/envs/MaxPlanck/lib/python3.11/site-packages/in_n_out/_store.py", line 804, in _exec
    result = func(**bound.arguments)
             ^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _close_app() missing 1 required positional argument: 'window'

In [24]:
from spatialdata.models import PointsModel
import numpy as np

points_layer = viewer.layers["calibration_points"]
image_points = points_layer.data
print(image_points)

sdata.points["calibration_points"] = PointsModel.parse(
    np.array(image_points)
)

[[12361.51636517  9967.83338582]
 [ 3543.83605132 10687.59528806]
 [ 3309.47665212   489.27618761]]


In [ ]:
from spatialdata.models import ShapesModel
from shapely.geometry import Polygon
import geopandas as gpd
import numpy as np

square_layer = viewer.layers["polygons"]
image_square = square_layer.data
print(image_square)

polygons = [Polygon(coords) for coords in image_square]
square_gdf = gpd.GeoDataFrame(
    {"shape_id": [f"square_{i}" for i in range(len(polygons))]},
    geometry=polygons
)

sdata.shapes["square"] = ShapesModel.parse(square_gdf)

In [25]:
pts = sdata.points["calibration_points"]
print(pts)

Dask DataFrame Structure:
                     x        y
npartitions=1                  
0              float64  float64
2                  ...      ...
Dask Name: frompandas, 1 expression
Expr=df


In [26]:
sdata

SpatialData object
├── Images
│     ├── 'image': DataArray[cyx] (3, 102276, 103963)
│     └── 'image_vis': DataArray[cyx] (3, 12784, 12995)
├── Points
│     └── 'calibration_points': DataFrame with shape: (<Delayed>, 2) (2D points)
└── Shapes
      └── 'tiles': GeoDataFrame shape: (20, 2) (2D shapes)
with coordinate systems:
    ▸ 'global', with elements:
        image (Images), image_vis (Images), calibration_points (Points), tiles (Shapes)

In [27]:
H = sdata.images["image"].data.shape[1]
print(H)

102276


In [ ]:

from dvpio.write import write_lmd

path_lmd = os.path.join(czi_dir, "lung5a.xml")

affine_transformation = np.array([
    [1,  0, 0],
    [0, -1, H],
    [0,  0, 1]
])

write_lmd(
    path_lmd,
    sdata.shapes["tiles"],
    calibration_points=sdata.points["calibration_points"],
    affine_transformation=affine_transformation
)

[12361.51636517 -9967.83338582]
[  3543.83605132 -10687.59528806]
[3309.47665212 -489.27618761]
